# 05 — Conformance Analysis

Compare the actual event log against the normative process model (Inductive Miner).
Detect deviations case by case. Flag compliance violations where `Start trip` precedes permit approval.

> **Note on the `Start trip` finding (from critical review):**
> - **Type A**: `Start trip` precedes `Permit SUBMITTED` — no permit initiated at time of travel
> - **Type B**: `Start trip` follows `Permit SUBMITTED` but precedes `Permit FINAL_APPROVED`

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pm4py
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from src.load_event_log import load_xes_log, convert_to_dataframe
from src.process_summary import get_case_durations
from pm4py.algo.filtering.log.variants import variants_filter

DATA_PATH   = Path('../data/raw/PermitLog.xes')
FIGURES_OUT = Path('../outputs/figures')
TABLES_OUT  = Path('../outputs/tables')
FIGURES_OUT.mkdir(parents=True, exist_ok=True)
TABLES_OUT.mkdir(parents=True, exist_ok=True)

In [2]:
# legacy=True returns an EventLog object required by PM4Py mining/replay algorithms
log_obj = load_xes_log(DATA_PATH, legacy=True)

# DataFrame version for all pandas analysis
df = convert_to_dataframe(log_obj)
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], utc=True)
df = df.sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

print(f'EventLog: {len(log_obj)} traces')
print(f'DataFrame: {len(df):,} events, {df["case:concept:name"].nunique():,} cases')

/opt/anaconda3/lib/python3.12/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/7065 [00:00<?, ?it/s]

EventLog: 7065 traces
DataFrame: 86,581 events, 7,065 cases


## 1. Rediscover the reference model

Inductive Miner (IMf, noise_threshold=0.2) on the top-30-variant log.
Same model as notebook 03 — reproduced here so this notebook is self-contained.

In [3]:
variants_dict = variants_filter.get_variants(log_obj)
top30 = dict(sorted(variants_dict.items(), key=lambda x: len(x[1]), reverse=True)[:30])
log_filtered = variants_filter.apply(log_obj, top30)
print(f'Filtered log (top 30 variants): {len(log_filtered)} traces')

net, im, fm = pm4py.discover_petri_net_inductive(log_filtered, noise_threshold=0.2)
print(f'Reference model — Places: {len(net.places)}, '
      f'Transitions: {len(net.transitions)}, Arcs: {len(net.arcs)}')

Filtered log (top 30 variants): 4055 traces
Reference model — Places: 32, Transitions: 42, Arcs: 90


## 2. Token-based replay — full log

In [4]:
from pm4py.algo.conformance.tokenreplay import algorithm as token_replay

replay_results = token_replay.apply(log_obj, net, im, fm)
print(f'Replay complete: {len(replay_results)} traces')

# Extract case IDs from the legacy EventLog object
case_ids = [trace.attributes['concept:name'] for trace in log_obj]

replay_df = pd.DataFrame({
    'case_id':          case_ids,
    'fitness':          [r['trace_fitness']    for r in replay_results],
    'missing_tokens':   [r['missing_tokens']   for r in replay_results],
    'remaining_tokens': [r['remaining_tokens'] for r in replay_results],
    'consumed_tokens':  [r['consumed_tokens']  for r in replay_results],
    'produced_tokens':  [r['produced_tokens']  for r in replay_results],
    'is_fit':           [r['trace_is_fit']     for r in replay_results],
})

replay_df.to_csv(TABLES_OUT / 'conformance_replay.csv', index=False)
print(f'Perfect fits  : {replay_df["is_fit"].sum():,} ({replay_df["is_fit"].mean()*100:.1f}%)')
print(f'Mean fitness  : {replay_df["fitness"].mean():.4f}')
print(f'Median fitness: {replay_df["fitness"].median():.4f}')
print()
print(replay_df[['fitness','missing_tokens','remaining_tokens']].describe().round(3))

replaying log with TBR, completed traces ::   0%|          | 0/1478 [00:00<?, ?it/s]

Replay complete: 7065 traces
Perfect fits  : 3,891 (55.1%)
Mean fitness  : 0.9602
Median fitness: 1.0000

        fitness  missing_tokens  remaining_tokens
count  7065.000        7065.000          7065.000
mean      0.960           0.965             1.017
std       0.059           1.594             1.648
min       0.571           0.000             0.000
25%       0.933           0.000             0.000
50%       1.000           0.000             0.000
75%       1.000           1.000             2.000
max       1.000          29.000            29.000


## 3. Fitness distribution

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(replay_df['fitness'], bins=40, color='#2c7bb6', edgecolor='white')
ax.axvline(replay_df['fitness'].mean(),   color='#d7191c', linestyle='--', linewidth=1.5,
           label=f'Mean = {replay_df["fitness"].mean():.3f}')
ax.axvline(replay_df['fitness'].median(), color='#1a9641', linestyle='--', linewidth=1.5,
           label=f'Median = {replay_df["fitness"].median():.3f}')
ax.set_xlabel('Trace fitness (0 = no fit, 1 = perfect)')
ax.set_ylabel('Number of cases')
ax.set_title('Token-Based Replay Fitness Distribution')
ax.legend()

ax2 = axes[1]
bucket_labels = ['< 0.5', '0.5–0.7', '0.7–0.9', '0.9–<1.0', '= 1.0 (perfect)']
bucket_counts = [
    (replay_df['fitness'] < 0.5).sum(),
    ((replay_df['fitness'] >= 0.5) & (replay_df['fitness'] < 0.7)).sum(),
    ((replay_df['fitness'] >= 0.7) & (replay_df['fitness'] < 0.9)).sum(),
    ((replay_df['fitness'] >= 0.9) & (replay_df['fitness'] < 1.0)).sum(),
    (replay_df['fitness'] == 1.0).sum(),
]
palette = ['#d7191c','#fdae61','#ffffbf','#a6d96a','#1a9641']
bars = ax2.bar(bucket_labels, bucket_counts, color=palette, edgecolor='white')
for bar, cnt in zip(bars, bucket_counts):
    pct = cnt / len(replay_df) * 100
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
             f'{cnt}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
ax2.set_ylabel('Number of cases')
ax2.set_title('Fitness Buckets')
ax2.set_xticklabels(bucket_labels, rotation=20, ha='right', fontsize=9)

plt.suptitle('Conformance — Token-Based Replay', fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'conformance_fitness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved conformance_fitness.png')

/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96206/2842996857.py:31: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax2.set_xticklabels(bucket_labels, rotation=20, ha='right', fontsize=9)


Saved conformance_fitness.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96206/2842996857.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Deviation patterns — activities concentrated in non-fitting traces

In [6]:
non_fit_cases = set(replay_df[~replay_df['is_fit']]['case_id'])
fit_cases     = set(replay_df[ replay_df['is_fit']]['case_id'])
print(f'Non-fitting: {len(non_fit_cases):,} ({len(non_fit_cases)/len(replay_df)*100:.1f}%)  '
      f'Fitting: {len(fit_cases):,}')

df_nf = df[df['case:concept:name'].isin(non_fit_cases)]
df_f  = df[df['case:concept:name'].isin(fit_cases)]

nf_cases_per_act = df_nf.groupby('concept:name')['case:concept:name'].nunique().rename('non_fit_cases')
f_cases_per_act  = df_f.groupby('concept:name')['case:concept:name'].nunique().rename('fit_cases')

dev = pd.concat([nf_cases_per_act, f_cases_per_act], axis=1).fillna(0).astype(int)
dev['total']        = dev['non_fit_cases'] + dev['fit_cases']
dev['pct_non_fit']  = (dev['non_fit_cases'] / dev['total'] * 100).round(1)
dev = dev.sort_values('pct_non_fit', ascending=False)
dev.to_csv(TABLES_OUT / 'conformance_deviation_activities.csv')

print('\nActivities most concentrated in non-fitting traces:')
print(dev.head(20).to_string())

Non-fitting: 3,174 (44.9%)  Fitting: 3,891

Activities most concentrated in non-fitting traces:
                                                  non_fit_cases  fit_cases  total  pct_non_fit
concept:name                                                                                  
Permit REJECTED by ADMINISTRATION                           130          0    130        100.0
Request For Payment REJECTED by MISSING                       7          0      7        100.0
Request For Payment REJECTED by EMPLOYEE                    203          0    203        100.0
Permit FOR_APPROVAL by ADMINISTRATION                         1          0      1        100.0
Permit FOR_APPROVAL by SUPERVISOR                             1          0      1        100.0
Request For Payment REJECTED by BUDGET OWNER                  4          0      4        100.0
Permit REJECTED by BUDGET OWNER                              52          0     52        100.0
Permit REJECTED by DIRECTOR                      

In [7]:
print('Missing token distribution across cases:')
tok = replay_df.groupby('missing_tokens').size().reset_index(name='n_cases')
tok['pct'] = (tok['n_cases'] / len(replay_df) * 100).round(1)
print(tok.to_string(index=False))

Missing token distribution across cases:
 missing_tokens  n_cases  pct
              0     3891 55.1
              1     1449 20.5
              2      957 13.5
              3      296  4.2
              4      229  3.2
              5       91  1.3
              6       57  0.8
              7       31  0.4
              8       25  0.4
              9       13  0.2
             10       12  0.2
             11        6  0.1
             12        2  0.0
             13        1  0.0
             15        1  0.0
             16        1  0.0
             17        1  0.0
             21        1  0.0
             29        1  0.0


## 5. Fitness by department

In [8]:
case_dept   = df.drop_duplicates('case:concept:name')[['case:concept:name','case:OrganizationalEntity']]
replay_dept = replay_df.merge(case_dept, left_on='case_id', right_on='case:concept:name')

dept_fitness = (
    replay_dept.groupby('case:OrganizationalEntity')
    .agg(
        n_cases      =('fitness','count'),
        mean_fitness =('fitness','mean'),
        pct_perfect  =('is_fit', lambda x: x.mean() * 100),
        mean_missing =('missing_tokens','mean'),
    )
    .round(3)
    .sort_values('mean_fitness')
    .reset_index()
)
dept_fitness.to_csv(TABLES_OUT / 'conformance_by_department.csv', index=False)

fig, ax = plt.subplots(figsize=(12, 8))
bar_colors = ['#d7191c' if f < 0.7 else '#fdae61' if f < 0.9 else '#1a9641'
              for f in dept_fitness['mean_fitness']]
bars = ax.barh(dept_fitness['case:OrganizationalEntity'].astype(str),
               dept_fitness['mean_fitness'], color=bar_colors)
ax.axvline(0.9, color='black', linestyle='--', linewidth=1, label='0.9 threshold')
for bar, fit, pct in zip(bars, dept_fitness['mean_fitness'], dept_fitness['pct_perfect']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{fit:.3f} ({pct:.0f}% perfect)', va='center', fontsize=7.5)
ax.set_xlabel('Mean trace fitness')
ax.set_title('Mean Conformance Fitness by Department', fontsize=12)
ax.set_xlim(0, 1.17)
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'conformance_by_department.png', dpi=150)
plt.show()
print('Saved conformance_by_department.png')

Saved conformance_by_department.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96206/2283604070.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Compliance violation: Start trip before permit approval

In [9]:
case_col = 'case:concept:name'
act_col  = 'concept:name'
time_col = 'time:timestamp'

def first_ts(activity):
    return (df[df[act_col] == activity]
            .groupby(case_col)[time_col].min()
            .rename(activity))

timeline = pd.concat([
    first_ts('Start trip'),
    first_ts('Permit SUBMITTED by EMPLOYEE'),
    first_ts('Permit FINAL_APPROVED by SUPERVISOR'),
    first_ts('Permit FINAL_APPROVED by DIRECTOR'),
], axis=1)
timeline.columns = ['start_trip','permit_submitted','final_sup','final_dir']
timeline['final_approved'] = timeline[['final_sup','final_dir']].min(axis=1)

has_trip = timeline['start_trip'].notna()

type_a = timeline[
    has_trip & (
        timeline['permit_submitted'].isna() |
        (timeline['start_trip'] < timeline['permit_submitted'])
    )
]
type_b = timeline[
    has_trip &
    timeline['permit_submitted'].notna() &
    (timeline['start_trip'] >= timeline['permit_submitted']) &
    (timeline['final_approved'].isna() | (timeline['start_trip'] < timeline['final_approved']))
]
compliant = timeline[
    has_trip &
    timeline['final_approved'].notna() &
    (timeline['start_trip'] >= timeline['final_approved'])
]

n_trip = has_trip.sum()
print(f'Cases with Start trip: {n_trip}')
print(f'  Type A (before permit submitted):   {len(type_a):4d}  {len(type_a)/n_trip*100:.1f}%')
print(f'  Type B (before final approval):     {len(type_b):4d}  {len(type_b)/n_trip*100:.1f}%')
print(f'  Compliant (after final approval):   {len(compliant):4d}  {len(compliant)/n_trip*100:.1f}%')

Cases with Start trip: 7065
  Type A (before permit submitted):    746  10.6%
  Type B (before final approval):      583  8.3%
  Compliant (after final approval):   5736  81.2%


In [10]:
# Violation rate by department
cases_with_trip = df[df[case_col].isin(timeline[has_trip].index)].drop_duplicates(case_col)
vd = cases_with_trip[[case_col,'case:OrganizationalEntity']].copy()
vd['type_a']    = vd[case_col].isin(type_a.index)
vd['type_b']    = vd[case_col].isin(type_b.index)
vd['compliant'] = vd[case_col].isin(compliant.index)

dept_viol = vd.groupby('case:OrganizationalEntity')[['type_a','type_b','compliant']].sum()
dept_viol['total']         = dept_viol.sum(axis=1)
dept_viol['pct_violation'] = ((dept_viol['type_a'] + dept_viol['type_b']) /
                               dept_viol['total'] * 100).round(1)
dept_viol = dept_viol.sort_values('pct_violation', ascending=False)
dept_viol.to_csv(TABLES_OUT / 'violation_by_department.csv')
vd[vd['type_a']].to_csv(TABLES_OUT / 'violation_type_a.csv', index=False)
vd[vd['type_b']].to_csv(TABLES_OUT / 'violation_type_b.csv', index=False)
print('Top 15 departments by violation rate:')
print(dept_viol.head(15).to_string())

Top 15 departments by violation rate:
                           type_a  type_b  compliant  total  pct_violation
case:OrganizationalEntity                                                 
organizational unit 65483       1       0          0      1          100.0
organizational unit 65477      17       1          4     22           81.8
organizational unit 65472       7       3         14     24           41.7
organizational unit 65480       1       2          6      9           33.3
organizational unit 65484       0       1          2      3           33.3
organizational unit 65457      62      36        266    364           26.9
organizational unit 65461      11      14         72     97           25.8
organizational unit 65459      48      89        475    612           22.4
organizational unit 65460      71      43        424    538           21.2
organizational unit 65456      96     112        784    992           21.0
organizational unit 65464      45      28        284    357   

In [11]:
# Stacked bar chart
plot_dept = dept_viol[dept_viol['total'] >= 20].sort_values('pct_violation', ascending=True)
labels = plot_dept.index.astype(str)

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(labels, plot_dept['compliant'], color='#1a9641', label='Compliant')
ax.barh(labels, plot_dept['type_b'], left=plot_dept['compliant'],
        color='#fdae61', label='Type B — before final approval')
ax.barh(labels, plot_dept['type_a'],
        left=plot_dept['compliant'] + plot_dept['type_b'],
        color='#d7191c', label='Type A — before permit submitted')
ax.set_xlabel('Cases with Start trip event')
ax.set_title('Compliance Profile by Department (≥ 20 travel cases)', fontsize=11)
ax.legend(loc='lower right')
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'compliance_by_department.png', dpi=150)
plt.show()
print('Saved compliance_by_department.png')

Saved compliance_by_department.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96206/1266067559.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Case duration by compliance group

In [12]:
dur = get_case_durations(df).set_index('case:concept:name')['duration_days']

groups = {
    'Compliant': dur[dur.index.isin(compliant.index)].dropna(),
    'Type B':    dur[dur.index.isin(type_b.index)].dropna(),
    'Type A':    dur[dur.index.isin(type_a.index)].dropna(),
}

print(f'{"Group":<12} {"n":>5} {"Median (d)":>11} {"Mean (d)":>9} {"p75":>7}')
print('-' * 48)
for name, g in groups.items():
    print(f'{name:<12} {len(g):>5} {g.median():>11.1f} {g.mean():>9.1f} {g.quantile(0.75):>7.1f}')

u1, p1 = stats.mannwhitneyu(groups['Compliant'], groups['Type A'], alternative='two-sided')
r1 = abs(1 - 2*u1/(len(groups['Compliant'])*len(groups['Type A'])))
print(f'\nMann-Whitney (Compliant vs Type A): p={p1:.4f}, |r|={r1:.3f}')

u2, p2 = stats.mannwhitneyu(groups['Compliant'], groups['Type B'], alternative='two-sided')
r2 = abs(1 - 2*u2/(len(groups['Compliant'])*len(groups['Type B'])))
print(f'Mann-Whitney (Compliant vs Type B): p={p2:.4f}, |r|={r2:.3f}')

Group            n  Median (d)  Mean (d)     p75
------------------------------------------------
Compliant     5736        77.4      91.1   120.9
Type B         583        45.3      62.6    98.2
Type A         746        46.7      78.0    98.7

Mann-Whitney (Compliant vs Type A): p=0.0000, |r|=0.280
Mann-Whitney (Compliant vs Type B): p=0.0000, |r|=0.341


In [13]:
palette_g = {'Compliant': '#1a9641', 'Type B': '#fdae61', 'Type A': '#d7191c'}
fig, ax = plt.subplots(figsize=(10, 5))
for name, g in groups.items():
    ax.hist(g.clip(upper=400), bins=40, alpha=0.6, color=palette_g[name],
            label=f'{name} (n={len(g)}, med={g.median():.0f}d)')
ax.set_xlabel('Case duration (days, clipped at 400)')
ax.set_ylabel('Number of cases')
ax.set_title('Case Duration by Compliance Group')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'compliance_duration.png', dpi=150)
plt.show()
print('Saved compliance_duration.png')

Saved compliance_duration.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96206/1503896255.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Summary

In [14]:
print('=== CONFORMANCE SUMMARY ===')
print(f'Model  : IMf Petri net (noise_threshold=0.2, top-30-variant log)')
print(f'Method : token-based replay on full log ({len(replay_df):,} cases)')
print(f'Mean fitness   : {replay_df["fitness"].mean():.4f}')
print(f'Median fitness : {replay_df["fitness"].median():.4f}')
print(f'Perfect fits   : {replay_df["is_fit"].sum():,} ({replay_df["is_fit"].mean()*100:.1f}%)')
print()
print('=== COMPLIANCE VIOLATIONS ===')
print(f'Type A (before permit submitted): {len(type_a):,} ({len(type_a)/n_trip*100:.1f}% of travel cases)')
print(f'Type B (before final approval):   {len(type_b):,} ({len(type_b)/n_trip*100:.1f}% of travel cases)')
print(f'Compliant:                        {len(compliant):,} ({len(compliant)/n_trip*100:.1f}% of travel cases)')
print()
print('=== DURATION BY COMPLIANCE GROUP ===')
for name, g in groups.items():
    print(f'  {name:<12}: median={g.median():.0f}d  mean={g.mean():.0f}d  n={len(g)}')

=== CONFORMANCE SUMMARY ===
Model  : IMf Petri net (noise_threshold=0.2, top-30-variant log)
Method : token-based replay on full log (7,065 cases)
Mean fitness   : 0.9602
Median fitness : 1.0000
Perfect fits   : 3,891 (55.1%)

=== COMPLIANCE VIOLATIONS ===
Type A (before permit submitted): 746 (10.6% of travel cases)
Type B (before final approval):   583 (8.3% of travel cases)
Compliant:                        5,736 (81.2% of travel cases)

=== DURATION BY COMPLIANCE GROUP ===
  Compliant   : median=77d  mean=91d  n=5736
  Type B      : median=45d  mean=63d  n=583
  Type A      : median=47d  mean=78d  n=746
